# 01 · The simplex method in matrix form
**Branch-and-Cut Workshop · UANL & Tecnológico de Monterrey** (Prof. Angélica Salazar, Prof. Vincent Boyer)

*Activity 2 · about 7 min.*

> **How to use this notebook.** Run the cells from top to bottom (Shift+Enter).
> Cells marked **✍️ TODO** are for you. Cells marked **🤖 AI prompt** contain a prompt you can paste into
> Gemini (the ✨ button in Colab), ChatGPT, Claude… **Always read and test the code the AI gives you.**

The simplex method walks from **extreme point to extreme point** of the feasible region, always to a better
neighbour, until no neighbour is better. Each extreme point is described by a **basis**: $m$ columns of
$[A \mid I]$ (the variables allowed to be non-zero). Split the columns into basic ($B$, variables $x_B$) and
non-basic ($H$, variables $x_H$), and multiply $B x_B + H x_H = b$ by $B^{-1}$:

$$x_B + B^{-1}H\,x_H = B^{-1}b, \qquad z = c_B B^{-1} b + (c_H - c_B B^{-1} H)\,x_H .$$

Everything is read from these equations:
* **The extreme point:** $x_H = 0$, so $x_B = B^{-1}b$ and $z$ is the constant.
* **Optimal** when no coefficient of the $z$ row is positive (maximization): increasing any $x_j \in x_H$ cannot help.
* Otherwise the variable with the largest coefficient **enters**. Increase it: each row says how much it can grow
  before its basic variable hits 0 (ratio test). The first one to hit 0 **leaves**.
* **New basis:** the entering variable takes the place of the leaving one, so one column of $B$ changes;
  compute the new $B^{-1}$ and write the new equations (real solvers update $B^{-1}$ instead of inverting it).

These rows are exactly the ones **Gomory cuts** are read from (Part 4).

**Starting point.** With $Ax \le b$ and $b \ge 0$, the origin is an extreme point: all slacks are basic, $B = I$.
(With $\ge$ or $=$ rows the origin may be infeasible: Phase I or big-$M$ finds a first basis. Not needed today.)

Every number below is an exact fraction (`fractions.Fraction`), so you can compare with the slides.

In [ ]:
# @title Setup and helper functions (run this cell; no need to read it)
# PuLP 4 with the open-source HiGHS solver (not pre-installed on Colab: installed on first run, ~20 s)
try:
    import pulp, highspy
    if int(pulp.__version__.split(".")[0]) < 4:
        raise ImportError("old PuLP")
except ImportError:
    import importlib, subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pulp[highs]>=4"], check=True)
    for _m in [m for m in sys.modules if m == "pulp" or m.startswith("pulp.")]:
        del sys.modules[_m]                       # forget an older PuLP loaded earlier in this session
    importlib.invalidate_caches()
    import pulp
SOLVER = pulp.HiGHS(msg=False)                    # every model in these notebooks is solved with HiGHS
print("PuLP", pulp.__version__, "+ HiGHS")


import math, re
from fractions import Fraction as Fr
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Math, Markdown

def lp_data(prob):
    """Read  max c x  s.t.  A x <= b,  x >= 0  from a PuLP model, in exact fractions.
    '>=' rows are multiplied by -1; a 'min' objective is turned into 'max -c x'."""
    xs = prob.variables()
    if (not xs or not prob.objective or all(prob.objective.get(v, 0) == 0 for v in xs)
            or all(con.get(v, 0) == 0 for con in prob.constraints() for v in xs)):
        raise ValueError(f"Model '{prob.name}' has no variables or no constraints with non-zero coefficients. "
                         "Did you fill in the ✍️ TODO lines (objective and constraints)?")
    q = lambda v: Fr(v).limit_denominator(10**6)
    sign = 1 if prob.sense == pulp.LpMaximize else -1
    c = [sign * q(prob.objective.get(v, 0)) for v in xs]
    A, b = [], []
    for con in prob.constraints():
        row, rhs = [q(con.get(v, 0)) for v in xs], -q(con.constant)
        if con.sense == pulp.LpConstraintGE:
            row, rhs = [-a for a in row], -rhs
        elif con.sense == pulp.LpConstraintEQ:
            raise ValueError("this small demo handles <= and >= constraints only")
        A.append(row); b.append(rhs)
    return [v.name for v in xs], c, A, b

def matmul(X, Y):
    return [[sum(X[i][k] * Y[k][j] for k in range(len(Y))) for j in range(len(Y[0]))] for i in range(len(X))]

def matinv(X):
    """Exact inverse of a square matrix of fractions (Gauss-Jordan)."""
    m = len(X)
    T = [[Fr(v) for v in row] + [Fr(int(i == k)) for k in range(m)] for i, row in enumerate(X)]
    for c in range(m):
        p = next(i for i in range(c, m) if T[i][c] != 0)
        T[c], T[p] = T[p], T[c]
        T[c] = [v / T[c][c] for v in T[c]]
        for i in range(m):
            if i != c:
                T[i] = [a - T[i][c] * b for a, b in zip(T[i], T[c])]
    return [row[m:] for row in T]

def nice(v, den=1000):
    """Float from the solver -> short fraction string, e.g. 3.3333333 -> '10/3'."""
    return str(Fr(v).limit_denominator(den))

# ---- LaTeX output (fractions shown as fractions) ----
def tex_num(q):
    q = Fr(q)
    if q.denominator == 1:
        return str(q.numerator)
    s = r"\tfrac{%d}{%d}" % (abs(q.numerator), q.denominator)
    return "-" + s if q < 0 else s

def tex_var(name):
    return re.sub(r"^([A-Za-z]+)_?(\d+)$", r"\1_{\2}", name)

def tex_mat(M):
    return r"\begin{pmatrix}" + r"\\ ".join(" & ".join(tex_num(v) for v in row) for row in M) + r"\end{pmatrix}"

def tex_col(v):
    return tex_mat([[a] for a in v])

def tex_expr(coefs, names):
    out = ""
    for a, nm in zip(coefs, names):
        a = Fr(a)
        if a == 0:
            continue
        sign = ("-" if a < 0 else "") if not out else (" - " if a < 0 else " + ")
        out += sign + ("" if abs(a) == 1 else tex_num(abs(a)) + " ") + tex_var(nm)
    return out or "0"

def show(*parts):
    display(Math(r"\qquad ".join(parts)))

def plot_model(prob, path=(), cuts=(), points=True, star=None, ax=None, title=""):
    """Feasible region of a 2-variable model, its integer points, a simplex path and cuts (a, beta, label)."""
    names, c, A, b = lp_data(prob)
    A, b = np.array(A, float), np.array(b, float)
    hi = 1.15 * max(b[i] / A[i, j] for i in range(len(b)) for j in range(2) if A[i, j] > 0)
    ax = ax or plt.subplots(figsize=(5.5, 4.5))[1]
    X, Y = np.meshgrid(np.linspace(0, hi, 500), np.linspace(0, hi, 500))
    ok = np.all([A[i, 0] * X + A[i, 1] * Y <= b[i] + 1e-9 for i in range(len(b))], axis=0)
    ax.contourf(X, Y, ok, levels=[0.5, 1], colors=["#0072CE"], alpha=0.15)
    xs = np.linspace(0, hi, 200)
    for i in range(len(b)):
        if A[i, 1] != 0:
            ax.plot(xs, (b[i] - A[i, 0] * xs) / A[i, 1], color="#003865", lw=1)
    if points:
        pts = [(i, j) for i in range(int(hi) + 1) for j in range(int(hi) + 1)
               if all(A[k, 0] * i + A[k, 1] * j <= b[k] + 1e-9 for k in range(len(b)))]
        ax.scatter(*zip(*pts), s=14, color="k", zorder=3)
    for k, (a, beta, label) in enumerate(cuts):
        a = [float(v) for v in a]
        ax.plot(xs, (float(beta) - a[0] * xs) / a[1], "--", lw=2, label=label)
    if len(path):
        P = np.array([[float(v) for v in p] for p in path])
        ax.plot(P[:, 0], P[:, 1], "-o", color="#E07B00", lw=2, zorder=4, label="simplex path")
    if star is not None:
        ax.scatter([float(star[0])], [float(star[1])], marker="*", s=220, color="#C8102E", zorder=5)
    ax.set_xlim(-0.2, hi); ax.set_ylim(-0.2, hi); ax.set_xlabel(names[0]); ax.set_ylabel(names[1])
    ax.set_title(title or prob.name)
    if cuts or len(path):
        ax.legend(fontsize=8, loc="upper right")
    return ax

## The model in PuLP
Both problems of the workshop are written with **PuLP**. Here is the explanation problem $E$:

In [ ]:
# E: the explanation problem (used on the slides)
def make_E(integer=False):
    cat = "Integer" if integer else "Continuous"
    E = pulp.LpProblem("E", pulp.LpMaximize)
    x1 = E.add_variable("x1", lowBound=0, cat=cat)
    x2 = E.add_variable("x2", lowBound=0, cat=cat)
    E += 5 * x1 + 8 * x2                    # objective
    E += x1 + x2 <= 6, "c1"
    E += 5 * x1 + 9 * x2 <= 45, "c2"
    return E

E = make_E()
print(E)

## The simplex method, step by step
The next cell defines `simplex_steps` (collapsed: open it if you are curious); the output is what matters.

In [ ]:
# @title simplex_steps: the simplex method in fractions (run; no need to read)
def simplex_steps(prob, verbose=True):
    """Revised simplex on the LP relaxation of prob, in exact fractions.
    Starts from the slack basis (x = 0). At every iteration it writes the system as
    x_B + B^-1 H x_H = B^-1 b, and z as a function of x_H, and reads everything from these equations:
    the extreme point (x_H = 0), the entering variable, the ratio test.
    Returns the final basis and the visited extreme points."""
    names, c, A, b = lp_data(prob)
    m, n = len(A), len(c)
    cols = names + [f"s{i+1}" for i in range(m)]
    M = [A[i] + [Fr(int(i == k)) for k in range(m)] for i in range(m)]      # [A | I]
    cost = c + [Fr(0)] * m
    basis = list(range(n, n + m))                                        # slacks basic: x = 0
    Binv = [[Fr(int(i == k)) for k in range(m)] for i in range(m)]
    path = []
    for it in range(100):
        xB = [sum(Binv[i][k] * b[k] for k in range(m)) for i in range(m)]
        x = [Fr(0)] * n
        for i, j in enumerate(basis):
            if j < n:
                x[j] = xB[i]
        path.append(x)
        cB = [cost[j] for j in basis]
        y = [sum(cB[i] * Binv[i][k] for i in range(m)) for k in range(m)]
        red = {j: cost[j] - sum(y[k] * M[k][j] for k in range(m)) for j in range(n + m) if j not in basis}
        z = sum(cb * v for cb, v in zip(cB, xB))
        B = [[M[i][j] for j in basis] for i in range(m)]
        nb = list(red)                                                   # non-basic columns (H)
        H = [[M[i][j] for j in nb] for i in range(m)]
        tab = matmul(Binv, M)                                            # B^-1 [A | I]
        if verbose:
            display(Markdown(f"#### Iteration {it}: extreme point x = ({', '.join(str(v) for v in x)}), z = {z}"))
            show(r"x_B = (" + ", ".join(tex_var(cols[j]) for j in basis) + ")",
                 r"x_H = (" + ", ".join(tex_var(cols[j]) for j in nb) + ")",
                 "B = " + tex_mat(B), "H = " + tex_mat(H), "B^{-1} = " + tex_mat(Binv))
            zrow = tex_expr([red[j] for j in nb], [cols[j] for j in nb])
            eqs = [tex_expr([1] + [tab[i][j] for j in nb], [cols[basis[i]]] + [cols[j] for j in nb])
                   + " &= " + tex_num(xB[i]) for i in range(m)]
            eqs.append("z &= " + tex_num(z) + ("" if zrow == "0" else (" " if zrow[0] == "-" else " + ") + zrow))
            display(Markdown(r"$x_B + B^{-1}H\,x_H = B^{-1}b$, and $z$ written with $x_H$ only:"))
            show(r"\begin{aligned}" + r"\\ ".join(eqs) + r"\end{aligned}")
            display(Markdown(r"$x_H = 0$ gives the extreme point: $x_B = B^{-1}b$ and $z$ = the constant:"))
            full = {cols[j]: Fr(0) for j in nb}
            full.update({cols[j]: xB[i] for i, j in enumerate(basis)})
            show(r"x_B = (" + ", ".join(tex_var(cols[j]) for j in basis) + ") = (" + ", ".join(tex_num(v) for v in xB) + ")",
                 r"x_H = (" + ", ".join(tex_var(cols[j]) for j in nb) + ") = (" + ", ".join("0" for _ in nb) + ")",
                 r"\Rightarrow\;" + ",\\; ".join(f"{tex_var(v)} = {tex_num(full[v])}" for v in cols),
                 "z = " + tex_num(z))
        enter = max(red, key=lambda j: (red[j], -j))
        if red[enter] <= 0:
            if verbose:
                display(Markdown("**No positive coefficient in the z row: increasing any non-basic variable "
                                 "decreases z. This extreme point is optimal.**"))
            break
        d = [tab[i][enter] for i in range(m)]
        ratios = {i: xB[i] / d[i] for i in range(m) if d[i] > 0}
        if not ratios:
            raise ValueError("unbounded LP")
        r = min(ratios, key=lambda i: (ratios[i], basis[i]))
        if verbose:
            e = tex_var(cols[enter])
            display(Markdown(f"${e}$ has the largest positive coefficient in the z row: **${e}$ enters**. "
                             f"Increase ${e}$ (the other non-basic variables stay 0) while every row keeps $x_B \\ge 0$:"))
            show(r",\qquad ".join(f"{tex_var(cols[basis[i]])} = {tex_num(xB[i])} - {tex_expr([d[i]], [cols[enter]])} \\ge 0"
                                   rf"\Rightarrow {e} \le {tex_num(v)}" for i, v in ratios.items()))
            display(Markdown(f"**${tex_var(cols[basis[r]])}$ reaches 0 first: it leaves**, "
                             f"${e}$ takes its place in the basis."))
        basis[r] = enter                                                 # swap one column of B
        Binv = matinv([[M[i][j] for j in basis] for i in range(m)])
    return dict(basis=[cols[j] for j in basis], B=B, Binv=Binv, xB=xB, x=x, z=z, path=path,
                cols=cols, M=M, A=A, b=b, names=names, idx=basis)

In [ ]:
res = simplex_steps(E)

In [ ]:
plot_model(E, path=res["path"], title="E: the simplex path along the extreme points"); plt.show()
E.solve(SOLVER)          # check with a real solver
print("PuLP/HiGHS:", {v.name: v.value() for v in E.variables()}, "z =", pulp.value(E.objective))

**Things to notice.**
* The path is $(0,0) \to (0,5) \to (\tfrac94, \tfrac{15}4)$: each step moves along an edge to a neighbouring extreme point.
* At the end, the **columns of $B$ are the columns of $x_1, x_2$** in $A$, and $B^{-1}$ is exactly the matrix on the slides.
* The final equations $x_B + B^{-1}H\,x_H = B^{-1}b$ come back in Part 4: **Gomory cuts are read from them**.

---
## ✍️ Activity 2 — simplex on the practice problem $P$
$$P:\quad \max\; 3x_1 + 4x_2 \quad \text{s.t.}\quad x_1 + 2x_2 \le 8,\;\; 2x_1 + x_2 \le 9,\;\; x_1, x_2 \ge 0$$

1. Write $P$ in PuLP: copy the `make_E` cell and change the coefficients (or edit the cell below).
2. Run `simplex_steps(P)` and follow the path on the plot.
3. **Post on Miro** (frame *2 – Simplex*): the optimal basis, $B$, $B^{-1}$, $x_B$ and $z$, and the list of
   extreme points visited.
4. In one sentence: **why** does the method stop where it stops?

In [ ]:
def make_P(integer=False):
    cat = "Integer" if integer else "Continuous"
    P = pulp.LpProblem("P", pulp.LpMaximize)
    x1 = P.add_variable("x1", lowBound=0, cat=cat)
    x2 = P.add_variable("x2", lowBound=0, cat=cat)
    P += 0 * x1 + 0 * x2               # ✍️ TODO: objective of P
    P += 0 * x1 + 0 * x2 <= 0, "c1"    # ✍️ TODO: first constraint
    P += 0 * x1 + 0 * x2 <= 0, "c2"    # ✍️ TODO: second constraint
    return P

P = make_P()
print(P)
if all(P.objective.get(v, 0) == 0 for v in P.variables()):
    print("⚠️ P is still empty: replace the 0 coefficients in the ✍️ TODO lines above, then run this cell again.")

In [ ]:
resP = simplex_steps(P)
plot_model(P, path=resP["path"], title="P: the simplex path"); plt.show()

## 🚀 If you have time
* Rounding the LP point of $P$ gives $(3, 2)$. Is it feasible? Is it optimal for the **integer** problem?
  (`make_P(integer=True).solve()` tells you.) This gap is the subject of Part 3.
* Change the objective of $P$ to $\max\, 4x_1 + 3x_2$. Does the path change? Does the optimal basis change?